In [ ]:
import h5py
import yaml
import numpy as np

with h5py.File("Data/train/nparticles_resampled.h5", "r") as f:
    for key in f.keys():
        print(f"{key}: {f[key].shape}")

In [ ]:
import h5py
import numpy as np
import yaml
import tensorflow as tf

# Using yaml 'mean' and 'std' values for normalization
# Z-Score Normalization / Standardization
with open("Data/train/nparticles_resampled.yaml", "r") as f:
    meta = yaml.safe_load(f)

def load_split(folder_path):
    h5_path = f"{folder_path}/nparticles_resampled.h5"
    
    with h5py.File(h5_path, "r") as hf:
        # Y (Labels)
        y_int = (hf['nparticles'][:] - 1).astype(np.int32).flatten()
        y = tf.keras.utils.to_categorical(y_int, num_classes=3)

        # X1 (Charge Matrix)
        c_mean = np.array(meta['charge']['mean'], dtype=np.float32)
        c_std  = np.array(meta['charge']['std'], dtype=np.float32)
        X1 = (hf['charge'][:] - c_mean) / c_std

        # X2 (Context Vector)
        # Pitches
        p_mean = np.array(meta['pitches']['mean'], dtype=np.float32)
        p_std  = np.array(meta['pitches']['std'], dtype=np.float32)
        norm_pitches = (hf['pitches'][:] - p_mean) / p_std
        
        # Angles
        phi_mean = np.array(meta['mean_phi']['mean'], dtype=np.float32)
        phi_std  = np.array(meta['mean_phi']['std'], dtype=np.float32)
        norm_phi = (hf['mean_phi'][:] - phi_mean) / phi_std

        theta_mean = np.array(meta['mean_theta']['mean'], dtype=np.float32)
        theta_std  = np.array(meta['mean_theta']['std'], dtype=np.float32)
        norm_theta = (hf['mean_theta'][:] - theta_mean) / theta_std

        # Categorical Flags
        raw_layer = hf['layer'][:].astype(np.float32)
        raw_bec   = hf['barrelec'][:].astype(np.float32)

        # Stack all
        X2 = np.hstack([norm_pitches, norm_phi, norm_theta, raw_layer, raw_bec])
        X = np.hstack([X1, X2])  
    return X, y

X_train, y_train = load_split("Data/train")
X_val, y_val     = load_split("Data/val")
X_test, y_test   = load_split("Data/test")

print(f"Loaded {len(y_train)} training samples.")
print(f"Loaded {len(y_val)} validation samples.")
print(f"Loaded {len(y_test)} test samples.")

In [ ]:
import numpy as np
import os

# Create a directory for the processed data
os.makedirs("Data/processed_data", exist_ok=True)

# Save Training Set
np.save("Data/processed_data/X_train.npy", X_train)
np.save("Data/processed_data/y_train.npy", y_train)

# Save Validation Set
np.save("Data/processed_data/X_val.npy", X_val)
np.save("Data/processed_data/y_val.npy", y_val)

# Save Test Set
np.save("Data/processed_data/X_test.npy", X_test)
np.save("Data/processed_data/y_test.npy", y_test)

print("All files saved to Data/processed_data/")